# 🌲 Forest: Bird Acoustic Monitoring & Species Classification in Nepal

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sauhardh/forest/blob/main/forest_colab.ipynb)

This notebook provides a complete, easy setup to run the **Forest** acoustic monitoring pipeline and train deep learning models on **Google Colab** with GPU acceleration.

### Highlights
- **Architecture**: Deep convolutional audio backbone (EfficientNet-V2-S / EfficientNet-B0)
- **Audio Processing**: 32 kHz normalization, PCEN / Log-Mel spectrograms, SpecAugment & Acoustic Mixup
- **Imbalance Handling**: Class-Balanced BCE Loss & Focal Loss for 300+ Himalayan bird species
- **Google Drive Persistence**: Auto-saves checkpoints and datasets to Google Drive so no data is lost on disconnect.

## Step 1: Check GPU Acceleration
Make sure you have selected a GPU runtime (**Runtime** → **Change runtime type** → **T4 GPU** or better).

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ WARNING: No GPU detected! Go to Runtime -> Change runtime type -> Select T4 GPU.")

## Step 2: Clone Repository & Install Dependencies
Clones the repository, configures Python paths, and installs required packages.

In [ ]:
import os
import sys
from pathlib import Path

# Clone repo if running in fresh Colab instance
if not Path("/content/forest").exists():
    !git clone https://github.com/sauhardh/forest.git /content/forest

%cd /content/forest
!git pull

# Ensure project and app directories are in sys.path
for p in ["/content/forest/app", "/content/forest", str(Path.cwd() / "app"), str(Path.cwd())]:
    if p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# Install requirements
!pip install -q -r requirements.txt
print("✓ Dependencies and python environment configured successfully!")

## Step 3 (Recommended): Mount Google Drive for Persistence
Mounting Google Drive ensures that checkpoints and processed datasets stay safe even if Colab restarts.

In [ ]:
#@title Google Drive Configuration
USE_GOOGLE_DRIVE = True #@param {type:"boolean"}
DRIVE_FOLDER_NAME = "forest_outputs" #@param {type:"string"}

import os
import sys
from pathlib import Path

for p in ["/content/forest/app", "/content/forest", str(Path.cwd() / "app"), str(Path.cwd())]:
    if p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_output_dir = Path("/content/drive/MyDrive") / DRIVE_FOLDER_NAME
    drive_output_dir.mkdir(parents=True, exist_ok=True)
    os.environ["FOREST_OUTPUT_DIR"] = str(drive_output_dir)
    print(f"✓ Data & Checkpoints will persist in Google Drive: {drive_output_dir}")
else:
    local_output_dir = Path("/content/forest/app/outputs")
    local_output_dir.mkdir(parents=True, exist_ok=True)
    os.environ["FOREST_OUTPUT_DIR"] = str(local_output_dir)
    print(f"Using local ephemeral storage: {local_output_dir}")

## Step 4: Dataset Setup
Choose how you want to prepare your dataset:
- **Option 1**: Extract an uploaded zip archive (e.g.  from your Google Drive or computer).
- **Option 2**: Run the automated data download and clip extraction pipeline directly on Colab.

In [ ]:
#@title Dataset Setup Options
DATA_SOURCE = "Check Existing / Drive" #@param ["Check Existing / Drive", "Unzip Archive", "Run Full Pipeline"]
ZIP_PATH = "/content/drive/MyDrive/forest_outputs.zip" #@param {type:"string"}

import os
import sys
import shutil
from pathlib import Path

# Ensure forest and app are in sys.path regardless of execution order
for p in ["/content/forest/app", "/content/forest", str(Path.cwd() / "app"), str(Path.cwd())]:
    if p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

from audio import BASE_OUTPUT_DIR, CLIPS_METADATA_PATH

print(f"Active Output Directory: {BASE_OUTPUT_DIR}")

if DATA_SOURCE == "Unzip Archive":
    if Path(ZIP_PATH).exists():
        print(f"Unpacking {ZIP_PATH} into {BASE_OUTPUT_DIR}...")
        shutil.unpack_archive(ZIP_PATH, BASE_OUTPUT_DIR)
        print("✓ Unpack complete!")
    else:
        print(f"❌ Zip file not found at: {ZIP_PATH}. Please check the path.")

elif DATA_SOURCE == "Run Full Pipeline":
    print("Running data collection & processing pipeline...")
    # 1. Split dataset
    !python app/audio/split_dataset.py
    # 2. Download audio
    !python app/audio/download_audio.py
    # 3. Normalize audio (32 kHz mono)
    !python app/audio/process/normalize_audio.py
    # 4. Extract clips
    !python app/audio/process/extract_clips.py
    print("✓ Pipeline run complete!")

# Check if clips metadata exists
if CLIPS_METADATA_PATH.exists():
    import pandas as pd
    df = pd.read_csv(CLIPS_METADATA_PATH)
    print(f"✓ Found clips metadata: {len(df)} clips across {df['species'].nunique()} species.")
else:
    print(f"ℹ️ clips_metadata.csv not found at {CLIPS_METADATA_PATH}.")
    print("If you have an archive on your computer/Drive, set DATA_SOURCE to 'Unzip Archive', or choose 'Run Full Pipeline'.")

## Step 5: Inspect Audio Spectrograms & Dataloaders
Test the dataset loader and inspect a generated spectrogram with PCEN / Mel Transform.

In [ ]:
import sys
from pathlib import Path
for p in ["/content/forest/app", "/content/forest", str(Path.cwd() / "app"), str(Path.cwd())]:
    if p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

import matplotlib.pyplot as plt
from audio import CLIPS_METADATA_PATH
from audio.model.dataset import BirdDataset

if CLIPS_METADATA_PATH.exists():
    val_dataset = BirdDataset(clips_csv=CLIPS_METADATA_PATH, split="val")
    print(f"Validation samples: {len(val_dataset)}")
    sample = val_dataset[0]
    spec = sample["spectrogram"].squeeze(0).numpy()
    species_name = val_dataset.idx_to_species[sample["label"].item()]

    plt.figure(figsize=(10, 4))
    plt.imshow(spec, origin="lower", aspect="auto", cmap="viridis")
    plt.title(f"Spectrogram for {species_name}")
    plt.xlabel("Time Frames")
    plt.ylabel("Mel Frequency Bins")
    plt.colorbar(format="%+2.0f dB")
    plt.tight_layout()
    plt.show()
else:
    print("Please set up dataset in Step 4 before visualizing.")

## Step 6: Train the Model
Train the model with PyTorch AMP (Automatic Mixed Precision) on the GPU.
Checkpoints will automatically save to your designated checkpoint folder (in Google Drive if enabled).

In [ ]:
#@title Training Hyperparameters
EPOCHS = 25 #@param {type:"integer"}
BATCH_SIZE = 32 #@param {type:"integer"}
LEARNING_RATE = 5e-4 #@param {type:"number"}
BACKBONE = "efficientnet_v2_s" #@param ["efficientnet_v2_s", "efficientnet_b0"]
USE_MIXUP = True #@param {type:"boolean"}

import sys
from pathlib import Path
for p in ["/content/forest/app", "/content/forest", str(Path.cwd() / "app"), str(Path.cwd())]:
    if p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

from audio.model.train import main as train_main
from audio.model.train import parse_args

cmd_args = [
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--lr", str(LEARNING_RATE),
    "--backbone", BACKBONE,
]
if not USE_MIXUP:
    cmd_args.append("--no-mixup")

parser = parse_args()
args = parser.parse_args(cmd_args)

# Start Training
train_main(args)

## Step 7: Export & Download Best Checkpoint
Your best model checkpoint is saved at .
If you enabled Google Drive in Step 3, it is already persisted in your Drive!

In [ ]:
import sys
from pathlib import Path
for p in ["/content/forest/app", "/content/forest", str(Path.cwd() / "app"), str(Path.cwd())]:
    if p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

from audio import CHECKPOINTS_DIR

best_ckpt = CHECKPOINTS_DIR / "best_model.pt"
if best_ckpt.exists():
    print(f"✓ Found checkpoint: {best_ckpt} ({best_ckpt.stat().st_size / (1024*1024):.1f} MB)")
    # If running in Colab and you want to download directly:
    # from google.colab import files
    # files.download(str(best_ckpt))
else:
    print(f"No checkpoint found at {best_ckpt}. Run training first.")